# Session 6 — pandas and Tabular Data

**Goal of this session:** work with the trial tables that every experiment produces.

*Python for Neuroscience, session 6 of 12.*

## Why this matters

Your signal is an array. Everything around it is a table: which condition each trial was, how fast the subject responded, whether they got it right, which subject it was.

pandas is the library for that table. If you have used a spreadsheet, you already know the shape. The difference is that here it is scriptable and reproducible.

## Building a DataFrame

A DataFrame is a table with named columns. The most common way to make one is from a dictionary, where each key is a column name.

Let's simulate 60 trials of a simple experiment with two conditions.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_trials = 60

conditions = rng.choice(["attend", "ignore"], size=n_trials)

# attended trials are a bit faster and a bit more accurate
rt = np.where(conditions == "attend",
              rng.normal(430, 60, n_trials),
              rng.normal(505, 75, n_trials))
correct = np.where(conditions == "attend",
                   rng.random(n_trials) < 0.92,
                   rng.random(n_trials) < 0.78)

trials = pd.DataFrame({
    "trial": np.arange(1, n_trials + 1),
    "condition": conditions,
    "rt_ms": rt.round(1),
    "correct": correct,
})

trials.head()

`.head()` shows the first five rows. There is a `.tail()` too, and `.shape` for the size.

In [ ]:
print(trials.shape, "= rows, columns")
print(trials.columns.tolist())
trials.describe()

`.describe()` gives you count, mean, standard deviation and the quartiles for every numeric column. It is the first thing to run on any table you did not create yourself, because it exposes impossible values fast.

## Getting at columns and rows

One column is a Series. Square brackets with a condition inside filters rows.

In [ ]:
print(trials["rt_ms"].mean())
print(trials["condition"].value_counts())

In [ ]:
fast_and_correct = trials[(trials["rt_ms"] < 400) & (trials["correct"])]
print(len(fast_and_correct), "trials were both fast and correct")
fast_and_correct.head()

The ampersand is the "and" for tables, and each condition needs its own brackets. That is a syntax quirk you will meet a few times before it sticks.

## groupby, the reason people use pandas

Split the table by condition, apply a calculation to each part, and get one row per group back. Three ideas, one line.

In [ ]:
summary = trials.groupby("condition").agg(
    mean_rt=("rt_ms", "mean"),
    sd_rt=("rt_ms", "std"),
    accuracy=("correct", "mean"),
    n=("trial", "count"),
).round(3)

summary

There is your result table. Attended trials came out faster and more accurate, which is what we built into the simulation, so the pipeline works.

## The visual

In [ ]:
import matplotlib.pyplot as plt

means = trials.groupby("condition")["rt_ms"].mean()
sems = trials.groupby("condition")["rt_ms"].sem()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(means.index, means.values, yerr=sems.values, capsize=8,
       color=["#2b6cb0", "#a0aec0"])
ax.set_ylabel("reaction time (ms)", fontsize=13)
ax.set_title("Mean reaction time by condition", fontsize=15)
ax.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

The bars are the group means and the whiskers are the standard error, which is the standard deviation divided by the square root of the sample size. Error bars are not optional. A bar chart without them tells you nothing about whether the difference is real.

## Bridging back to the signal

Tables and signals meet when you cut a signal into trials and summarise each one. Here we take our toy signal, chop it into 20 segments, and put one row per segment into a DataFrame.

In [ ]:
import numpy as np


def generate_toy_signal(duration=2.0, sampling_rate=500.0, noise_level=0.5,
                        freq=10.0, amplitude=1.0, seed=0):
    """A toy oscillatory signal: one sine wave plus white noise.

    This is not a recording. It is a stand-in that behaves enough like an
    alpha rhythm to practise on. Returns the time axis and the signal.
    """
    rng = np.random.default_rng(seed)
    t = np.arange(0, duration, 1 / sampling_rate)
    signal = amplitude * np.sin(2 * np.pi * freq * t)
    signal = signal + noise_level * rng.standard_normal(t.size)
    return t, signal

In [ ]:
t, signal = generate_toy_signal(duration=20.0, sampling_rate=500.0, noise_level=0.5)

n_segments = 20
segments = np.array_split(signal, n_segments)

segment_table = pd.DataFrame({
    "segment": np.arange(1, n_segments + 1),
    "mean": [s.mean() for s in segments],
    "sd": [s.std() for s in segments],
    "peak": [np.abs(s).max() for s in segments],
}).round(3)

segment_table.head()

That table is now the bridge between a raw recording and any statistics you want to run on it. Every trial-based analysis in neuroscience has this step in it somewhere.

## Try it yourself

Group `trials` by `correct` instead of `condition` and look at the reaction times. Correct responses being slower or faster than errors is a real and much-argued-about effect.

**Next session:** making the figures properly.